# Aprendizaje no supervisado — Segmentos de canciones (K-Means)
### Evaluación Parcial N°2 — MLY1101 Machine Learning

**Objetivo de negocio:** la distribuidora no tiene un solo tipo de canción en su catálogo. Si se descubren **grupos de canciones con un sonido parecido**, se puede entender qué segmentos existen, qué géneros los componen y qué tan populares son, para **diseñar estrategias de promoción distintas por segmento** en vez de una sola para todo el catálogo.

**Técnica elegida: K-Means** sobre las audio features.
- No hay una etiqueta que predecir: se busca estructura en los datos (aprendizaje no supervisado).
- Es simple, escala bien a ≈ 90.000 canciones y sus grupos se interpretan fácilmente mirando el promedio de cada variable por cluster.
- **Idea clave:** el clustering usa **solo el sonido** de la canción. El género y `popularity` **no** entran al algoritmo; se usan después para interpretar los grupos. Así se puede comprobar si los grupos "de sonido" coinciden con géneros reales y con la popularidad.

## 8. Clustering

### 8.1 Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_style('whitegrid')
RANDOM_STATE = 42

df = pd.read_csv('../data/processed/spotify_tracks_clean.csv')
genre_cols = [c for c in df.columns if c.startswith('genre_')]
print(df.shape)

### 8.2 Selección y estandarización de variables

K-Means usa distancias, por lo que las variables deben estar en la misma escala (por ejemplo `tempo` va de 0 a 240 y `energy` de 0 a 1). Se **estandarizan** (media 0, desviación 1) para que ninguna domine.

In [ ]:
# Variables continuas del "sonido" de la canción. Se excluyen key, mode y time_signature
# (son categóricas) y duration_ms; el género y popularity NO entran al clustering:
# se usan después para interpretar los grupos.
audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
                  'instrumentalness', 'liveness', 'valence', 'tempo']

scaler = StandardScaler()
X = scaler.fit_transform(df[audio_features])
print('Variables usadas:', len(audio_features), '| canciones:', X.shape[0])

### 8.3 Elección del número de clusters (k)

Se prueban k de 2 a 10 con dos criterios:
- **Método del codo:** la inercia (distancia interna de los grupos) siempre baja al aumentar k; se busca dónde deja de bajar rápido.
- **Coeficiente de silueta:** mide qué tan bien separado está cada punto de otros grupos (de -1 a 1; más alto es mejor).

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
muestra = rng.choice(len(X), 10000, replace=False)   # silhouette sobre muestra (es costoso en 90.000 filas)

ks = range(2, 11)
inercia, silueta = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(X[muestra], km.labels_[muestra]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inercia, marker='o')
axes[0].set_title('Método del codo'); axes[0].set_xlabel('k (número de clusters)'); axes[0].set_ylabel('Inercia')
axes[1].plot(list(ks), silueta, marker='o', color='darkorange')
axes[1].set_title('Coeficiente de silueta'); axes[1].set_xlabel('k (número de clusters)'); axes[1].set_ylabel('Silueta')
plt.tight_layout()
plt.savefig('../images/clustering_codo_silueta.png', dpi=100)
plt.show()

print(pd.DataFrame({'k': list(ks), 'inercia': np.round(inercia), 'silueta': np.round(silueta, 3)}).to_string(index=False))

**Decisión: k = 6.**
- El codo **no marca un quiebre claro** (los datos de audio son un continuo, no grupos perfectamente separados).
- La silueta es máxima con k = 2 (0,26), pero eso solo separa "canciones intensas" de "canciones suaves": poco útil para el negocio. Después de k = 2 la silueta es baja (≈ 0,16-0,20) y tiene un máximo local en **k = 6-7**.
- Con k = 6 los grupos son **interpretables y se alinean con géneros reales** (ver 8.5), y ninguno es demasiado pequeño como para ser inútil.
- Se reconoce que una silueta de ≈ 0,19 indica grupos **con solapamiento**: los segmentos son tendencias, no categorías cerradas.

### 8.4 Ajuste del modelo final (K-Means con k = 6)

In [ ]:
K = 6
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)
df['cluster'] = kmeans.labels_
print('Silueta con k =', K, ':', round(silhouette_score(X[muestra], kmeans.labels_[muestra]), 3))
df['cluster'].value_counts().sort_index()

### 8.5 Interpretación de los clusters

El mapa de calor muestra, para cada cluster, cuántas desviaciones estándar está por encima (rojo) o por debajo (azul) del promedio general en cada variable.

In [ ]:
perfil = df.groupby('cluster')[audio_features].mean()
z = (perfil - df[audio_features].mean()) / df[audio_features].std()   # cuántas desviaciones sobre/bajo el promedio

plt.figure(figsize=(10, 4.5))
sns.heatmap(z, annot=True, fmt='.1f', cmap='coolwarm', center=0)
plt.title('Perfil de cada cluster (desviaciones respecto al promedio general)')
plt.tight_layout()
plt.savefig('../images/clustering_perfil.png', dpi=100)
plt.show()

resumen = df.groupby('cluster').agg(canciones=('track_id', 'size'),
                                    popularity_media=('popularity', 'mean'),
                                    pct_top=('popularity', lambda s: (s > 70).mean() * 100),
                                    pct_explicit=('explicit', lambda s: s.mean() * 100)).round(1)
resumen

In [ ]:
# Géneros sobrerrepresentados en cada cluster (lift = proporción en el cluster / proporción en todo el dataset)
base = df[genre_cols].mean()
for c in range(K):
    prop = df.loc[df['cluster'] == c, genre_cols].mean()
    lift = (prop / base)[prop > 0.02].sort_values(ascending=False).head(4)
    print(f'Cluster {c}:', ', '.join(f"{g.replace('genre_', '')} (x{v:.1f})" for g, v in lift.items()))

**Segmentos encontrados** (los números de cluster pueden cambiar si se usa otra versión de scikit-learn; lo que se mantiene es el perfil):

| Cluster | Perfil de audio | Géneros característicos | Canciones | Popularity media | % top (>70) |
|---|---|---|---|---|---|
| 0 | **Intensas y agresivas**: energía y tempo altos, valence baja, poco acústicas | death-metal, grindcore, black-metal, metalcore | 24.302 | 33,3 | 2,6% |
| 1 | **En vivo**: `liveness` muy alta (+2,6 desv.) | pagode, sertanejo, samba, mpb | 6.639 | 34,8 | 1,4% |
| 2 | **Acústicas y suaves**: acousticness alta, energía baja | tango, honky-tonk, romance, jazz | 19.471 | 33,7 | 2,7% |
| 3 | **Bailables y alegres**: danceability y valence altas | reggaeton, latino, reggae, dancehall | 31.186 | 33,8 | **4,7%** |
| 4 | **Habladas**: `speechiness` extrema (+6,7 desv.), 56% explícitas | comedy (76% del cluster) | 1.052 | 24,6 | 0,5% |
| 5 | **Ambientales e instrumentales**: muy baja energía y loudness, alta instrumentalness | new-age, sleep, ambient, classical | 7.090 | 28,6 | 0,4% |

**Hallazgos:**
1. K-Means, sin conocer el género, **redescubrió agrupaciones de géneros coherentes** (por ejemplo, metal junto con metal, música para dormir junto con ambient). Eso valida que los segmentos tienen sentido musical.
2. El cluster 4 (comedy) es muy pequeño y muy distinto: es más bien contenido hablado que música.

### 8.6 Relación de los clusters con la popularidad

In [ ]:
orden = df.groupby('cluster')['popularity'].mean().sort_values(ascending=False).index

plt.figure(figsize=(9, 4.5))
sns.boxplot(x='cluster', y='popularity', data=df, order=orden)
plt.axhline(df['popularity'].mean(), color='red', linestyle='--', label='Media general')
plt.title('Popularity por cluster')
plt.legend()
plt.tight_layout()
plt.savefig('../images/clustering_popularity.png', dpi=100)
plt.show()

# ¿Cuánta variabilidad de popularity explica el cluster? (eta cuadrado)
ss_entre = sum(len(g) * (g['popularity'].mean() - df['popularity'].mean()) ** 2 for _, g in df.groupby('cluster'))
ss_total = ((df['popularity'] - df['popularity'].mean()) ** 2).sum()
print(f'El cluster explica {ss_entre / ss_total:.1%} de la variabilidad de popularity (eta²)')

**Interpretación:**
- La popularidad promedio es parecida en los clusters 0-3 (≈ 33-35). Solo los clusters 4 (habladas) y 5 (ambientales) son claramente menos populares (≈ 25 y 29).
- El cluster **explica solo ≈ 0,7% de la variabilidad de `popularity`**: el sonido por sí solo casi no determina qué tan popular es una canción. Esto es **coherente con el EDA y con el modelo supervisado** (R² de 0,45 aun usando género), y refuerza que la popularidad depende de factores externos (artista, promoción, fecha).
- Donde sí hay diferencia es en las **canciones top**: el cluster 3 (bailables y alegres) tiene **4,7%** de canciones con popularity > 70, frente a 0,4-0,5% en los clusters 4 y 5. Es decir, el sonido bailable/alegre se asocia a una mayor probabilidad de éxito, aunque no lo garantiza.

### 8.7 Visualización de los clusters en 2D (PCA)

Para poder dibujar 9 variables en un plano se usa PCA (reducción de dimensionalidad). Es solo para visualizar; el clustering se hizo con las 9 variables.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X)
vista = rng.choice(len(X), 10000, replace=False)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[vista, 0], y=coords[vista, 1], hue=df['cluster'].values[vista],
                palette='tab10', s=10, alpha=0.6)
plt.xlabel(f'Componente 1 ({pca.explained_variance_ratio_[0]:.0%} de la varianza)')
plt.ylabel(f'Componente 2 ({pca.explained_variance_ratio_[1]:.0%} de la varianza)')
plt.title('Clusters de canciones proyectados en 2D (PCA, muestra de 10.000)')
plt.legend(title='Cluster', markerscale=2)
plt.tight_layout()
plt.savefig('../images/clustering_pca.png', dpi=100)
plt.show()

Los grupos se ven **continuos y con solapamiento** (consistente con la silueta ≈ 0,19), salvo los clusters más extremos (5 y 4), que se separan mejor.

### 8.8 Guardado del modelo y de las asignaciones

In [ ]:
joblib.dump(scaler, '../models/scaler_clustering.joblib')
joblib.dump(kmeans, '../models/kmeans.joblib')
df[['track_id', 'cluster']].to_csv('../data/processed/clusters_canciones.csv', index=False)

## 9. Conclusiones y recomendaciones de la etapa no supervisada

1. El catálogo se puede dividir en **6 segmentos de sonido interpretables**, que coinciden con familias de géneros.
2. El sonido **no determina la popularidad** (el cluster explica ≈ 0,7%), pero sí cambia la probabilidad de ser un éxito: el segmento **bailable y alegre** concentra la mayor proporción de canciones top.
3. **Recomendaciones para la distribuidora:**
   - Usar los segmentos para **diferenciar campañas**: no esperar la misma popularidad de una pista ambiental o hablada que de una bailable; fijar metas de reproducciones por segmento.
   - Para el segmento **bailable y alegre** (el de mayor % de top), priorizar la inversión en promoción en playlists masivas.
   - Para los segmentos **de nicho** (ambientales, habladas, en vivo), apuntar a playlists temáticas (sueño, estudio, humor, música regional) en vez de campañas generales.
   - Combinar este análisis con el modelo supervisado: el cluster y el género permiten **comparar una canción contra su propio segmento**, no contra todo el catálogo.
4. **Limitaciones:** silueta baja (grupos solapados) y solo se usó información de audio; se recomienda repetir el análisis si se incorporan artista o fecha de lanzamiento.